# Finance Agent Demo

The Finance Agent is the first of 7 specialist agents. It inherits from `SpecialistAgent`
(`backend/agents/base_agent.py`), which gives every specialist the same governance guarantees
for free: every tool call is timed and logged as a `ToolCallRecord` regardless of what the
tool does internally, and the final output is always a validated `AgentBriefing`.

## Its 7 tools

| Tool | Computes | Data reality |
|---|---|---|
| `calculate_margin_trend` | Monthly contribution margin (`price - freight_value`) | No COGS field in source — freight is the only real per-order cost signal, flagged as a proxy |
| `detect_revenue_anomalies` | Daily revenue z-score outliers | Computed directly |
| `payment_failure_rate` | % of `canceled`/`unavailable` orders | Proxy — no explicit payment-failure field exists |
| `calculate_cogs` | — | **Explicitly returns "not available"** rather than fabricating a number |
| `cash_flow_forecast` | Linear-trend projection of net cash-in | Simple trend estimate, not a full AR/AP model |
| `refund_impact_analysis` | Payment value at risk from canceled orders | Proxy — no explicit refund field |
| `revenue_concentration` | Top-10 seller share + Herfindahl-Hirschman Index | Computed directly |

Every `Finding` the agent returns carries a `confidence` score and a `source` field naming the
exact tool that produced it — proxy-based tools get lower confidence than directly-computed
ones. This is the traceability requirement from the project's governance spec, enforced in
code rather than left to convention.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.finance import FinanceAgent
from backend.db import DatabricksClient

db = DatabricksClient()
agent = FinanceAgent(db)

## Run the agent against live Delta tables

In [2]:
briefing = agent.run("How is our financial health looking?")

print(f"Agent: {briefing.agent.value}")
print(f"Execution time: {briefing.execution_time_ms:.0f}ms")
print(f"Tool calls: {len(briefing.tool_calls)} ({sum(1 for t in briefing.tool_calls if t.success)} succeeded)")

Agent: Finance
Execution time: 13790ms
Tool calls: 7 (7 succeeded)


## Inspect the findings

Each `Finding` is what would get surfaced in the boardroom UX — a claim, its source, a confidence score, and a severity level (`info` / `warning` / `critical`).

In [3]:
for f in briefing.findings:
    print(f"[{f.severity.upper()}] (confidence {f.confidence:.2f}) {f.claim}")
    print(f"  source: {f.source}\n")

[INFO] (confidence 0.75) Contribution margin is improving across the last 12 months
  source: calculate_margin_trend

[WARNING] (confidence 0.85) 13 daily revenue anomalies detected out of 614 days analyzed
  source: detect_revenue_anomalies

[INFO] (confidence 0.60) Order failure rate (canceled/unavailable proxy) is 1.24% of 99441 orders
  source: payment_failure_rate

[WARNING] (confidence 1.00) COGS cannot be calculated - source data has no unit cost field
  source: calculate_cogs

[INFO] (confidence 0.55) Next month's projected cash-in is 1177859.93
  source: cash_flow_forecast

[INFO] (confidence 0.60) 1.68% of total payment value (269735.11) is at risk from canceled/unavailable orders
  source: refund_impact_analysis

[INFO] (confidence 0.90) Revenue concentration is low - top 10 sellers hold 13.2% of revenue (HHI 36.0)
  source: revenue_concentration



## Inspect a raw tool call

This is what `_call_tool()` captured automatically — timing and success/failure, with no extra code in the tool itself.

In [4]:
tc = briefing.tool_calls[0]
print(f"tool: {tc.tool_name}")
print(f"success: {tc.success}")
print(f"execution_time_ms: {tc.execution_time_ms:.1f}")
print(f"output_summary: {tc.output_summary[:200]}")

tool: calculate_margin_trend
success: True
execution_time_ms: 3364.5
output_summary: {'monthly': [{'month': '2017-10-01T00:00:00.000Z', 'revenue': '660179.62', 'freight_cost': '104576.41', 'contribution_margin': '555603.21', 'margin_pct': 84.16}, {'month': '2017-11-01T00:00:00.000Z', 


## Next

This briefing is exactly what the boss agent consumes — see `03_boss_agent_demo.ipynb` for how it gets selected, invoked, and synthesized into a board recommendation.